### Sample 10% of videos from outputs folder

In [2]:
import os
import random
import math

In [ ]:


base_dir = '/Users/eveyhuang/Documents/NICO/gemini_code/outputs'

# Collect all JSON files and map them to their video file names
video_file_map = {}
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith('.json') and not file.startswith('all_') and not file.startswith('verbal_'):
            video_name = file.replace('.json', '')
            full_path = os.path.join(root, file)
            if video_name not in video_file_map:
                video_file_map[video_name] = []
            video_file_map[video_name].append(full_path)

# Get all unique video names
all_video_names = list(video_file_map.keys())
print(f"Total unique video names found: {len(all_video_names)}")
n_sample = max(1, math.ceil(0.1 * len(all_video_names)))  # At least 1

print(f"Number of videos to sample: {n_sample}")
# Randomly sample 10% of video names
random.seed(42)  # For reproducibility
sampled_video_names = random.sample(all_video_names, n_sample)

# Get all file paths for the sampled videos
sampled_file_paths = []
for name in sampled_video_names:
    sampled_file_paths.extend(video_file_map[name])

# Print or save the sampled file paths
print("Sampled JSON files for verification:")
for path in sampled_file_paths:
    print(path)

# Optionally, save to a text file
with open('sampled_json_files.txt', 'w') as f:
    for path in sampled_file_paths:
        f.write(path + '\n')

Total unique video names found: 781
Number of videos to sample: 79
Sampled JSON files for verification:
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MND/output_2021_04_22_MND_S6/Breakout_Room_4_Part_2_2021_04_22_13_14_53/Breakout_Room_4_Part_2_2021_04_22_13_14_53_chunk6.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021ABI/output_2021_05_21_ABI_S5/bot5p3_Room_5_Zoom_Meeting_5_21_2021_10_59_19_AM/bot5p3_Room_5_Zoom_Meeting_5_21_2021_10_59_19_AM_chunk3.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021ABI/output_2021_05_20_ABI_S4/bot2p2_Zoom_Meeting_2021_05_20_12_37_07/bot2p2_Zoom_Meeting_2021_05_20_12_37_07_chunk3.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MZT/output_2021_10_01_MZT_S1/B1.2_Zoom_Meeting_Room_1_2021_10_01_11_07_45/B1.2_Zoom_Meeting_Room_1_2021_10_01_11_07_45.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021SLU/output_2021_06_10_SLU_S5/botB2_2021_06_10_12_35_06/botB2_2021_06_10_12_35_06_chunk2.json
/Users/eveyh

In [9]:
import json
import os
import re
sampled_dict = {}

with open('sampled_json_files.txt', 'r') as f:
    file_paths = [line.strip() for line in f if line.strip()]

def extract_key_and_subkey(path):
    # Get the part after '/outputs/'
    rest = path.split('/outputs/')[1]
    parts = rest.split(os.sep)
    key = os.path.join(parts[0], parts[1])
    sub_key = os.path.join(*parts[2:])  # Join everything after the key
    return key, sub_key

def find_verbal_annotations(data, file):
    for dt in file:
        if dt["transcript"] == data["transcript"]:
            data["annotations"] = dt["annotations"]
    return data


for path in file_paths:
    # Extract key after 'output_'
    try:
        key, sub_key = extract_key_and_subkey(path)
        
    except Exception as e:
        print(f"Skipping {path}: {e}")
        continue
    
    folder, filename = os.path.split(path)
    filename = re.sub(r'_chunk\d+(?=\.json)', '', filename)
    # Load JSON and sample a value
    try:
        with open(path, 'r') as jf:
            data = json.load(jf)
        with open(os.path.join(folder, 'all_'+filename), 'r') as f:
            all_data = json.load(f)
        if isinstance(data, list) and data:
            sampled_value = random.choice(data)
        elif isinstance(data, dict) and data:
            filtered = [ann for ann in data["meeting_annotations"] if ann["speaking duration"] > 15]
            if filtered:
                sample_size = min(3, len(filtered))
                sampled_value = random.sample(filtered, sample_size)
                for sample in sampled_value:
                    sample = find_verbal_annotations(sample, all_data)
            else:
                # Fallback to any annotation if none meet the criteria
                sampled_value = None
        else:
            sampled_value = data
    except Exception as e:
        print(f"Error reading {path}: {e}")
        continue

    if key not in sampled_dict:
        sampled_dict[key] = {}
    sampled_dict[key][sub_key] = sampled_value

# Save to a new JSON file
with open('sampled_verification.json', 'w') as out_f:
    json.dump(sampled_dict, out_f, indent=2)

Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MZT/output_2021_10_01_MZT_S1/B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18/B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S3/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44_chunk3.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S3/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44_chunk4.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S6/6_Theory_and_Expt_Zoom_Meeting_2020_11_05_10_28_39/6_Theory_and_Expt_Zoom_Meeting_2020_11_05_10_28_39_chunk1.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_06_NES_S7/1_beyond_co2

In [10]:
import pandas as pd

# Flatten sampled_verification for DataFrame
rows = []
for folder, files in sampled_dict.items():
    for file, samples in files.items():
        # samples can be a list or None
        if samples is None:
            rows.append({'folder': folder, 'file': file})
        elif isinstance(samples, list):
            for sample in samples:
                row = {'folder': folder, 'file': file}
                if isinstance(sample, dict):
                    row.update(sample)
                rows.append(row)
        elif isinstance(samples, dict):
            row = {'folder': folder, 'file': file}
            row.update(samples)
            rows.append(row)
        else:
            row = {'folder': folder, 'file': file, 'value': samples}
            rows.append(row)

df = pd.DataFrame(rows)
df.to_excel('sampled_verification.xlsx', index=False)

# Importing Evey's request

In [1]:
from pathlib import Path
BASE_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/outputs").expanduser()
RANDOM_SEED = 3839
OUT_CSV = BASE_DIR.parent / "sampling" / "sample_balanced.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [2]:
def find_all_gm_v4_files(base_dir: Path):
    return sorted(base_dir.rglob("all_gm_v4*.json"))

files = find_all_gm_v4_files(BASE_DIR)
print("Files found:", len(files))
assert files, "No all_gm_v4*.json files found under BASE_DIR; double-check the path."

Files found: 213


In [3]:
import json, pandas as pd
from typing import Any, Dict, List

def load_json_records(fp: Path) -> List[Dict[str, Any]]:
    try:
        with fp.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for v in data.values():
                if isinstance(v, list):
                    return v
    except json.JSONDecodeError:
        pass
    # NDJSON fallback
    recs = []
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    recs.append(obj)
            except json.JSONDecodeError:
                continue
    return recs

def extract_codes_from_item(item: dict) -> List[str]:
    # If labels/codes list exists, use it
    for key in ("labels", "codes"):
        if key in item and isinstance(item[key], list):
            return [str(x) for x in item[key] if isinstance(x, (str, int, float))]
    # Otherwise from annotations where score>0 (or value>0)
    ann = item.get("annotations")
    codes = []
    if isinstance(ann, dict):
        for k, v in ann.items():
            if isinstance(v, (bool, int, float)):
                if bool(v): codes.append(str(k))
            elif isinstance(v, dict):
                if "score" in v:
                    try:
                        if float(v["score"]) > 0: codes.append(str(k))
                    except Exception:
                        if bool(v["score"]): codes.append(str(k))
                elif "value" in v:
                    try:
                        if float(v["value"]) > 0: codes.append(str(k))
                    except Exception:
                        if bool(v["value"]): codes.append(str(k))
                elif any(bool(x) for x in v.values()):
                    codes.append(str(k))
    return codes

def infer_conference_from_path(p: Path, anchor="outputs") -> str:
    parts = list(p.parts)
    return parts[parts.index(anchor)+1] if anchor in parts and parts.index(anchor)+1 < len(parts) else "unknown"

def build_dataframe(file_paths: List[Path]) -> pd.DataFrame:
    rows = []
    for fp in file_paths:
        for i, rec in enumerate(load_json_records(fp)):
            rows.append({
                "conference": infer_conference_from_path(fp),
                "json_file": str(fp),
                "record_idx": i,
                "codes": extract_codes_from_item(rec),
                "speaker": rec.get("speaker"),
                "timestamp": rec.get("timestamp"),
                "transcript": rec.get("transcript"),
                "start_time": rec.get("start_time"),
                "end_time": rec.get("end_time"),
                "speaking_duration_raw": rec.get("speaking duration") or rec.get("speaking_duration") or rec.get("speaking_duration_sec"),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=["conference","json_file","record_idx","codes","speaker","timestamp","transcript","start_time","end_time","speaking_duration_raw"])
    return df

df = build_dataframe(files)
df["n_codes"] = df["codes"].apply(lambda x: len(x) if isinstance(x, list) else 0)
print("Utterances loaded:", len(df), "| conferences:", sorted(df["conference"].unique()))
print("Rows with empty codes:", int((df["n_codes"] == 0).sum()))

Utterances loaded: 30558 | conferences: ['2020NES', '2021ABI', '2021CMC', '2021MND', '2021MZT', '2021NES', '2021SLU', '2022MND']
Rows with empty codes: 4818


In [4]:
from datetime import datetime
import re
import math

def _parse_hhmm_or_mmss_to_seconds(s: str) -> float:
    if not isinstance(s, str) or ":" not in s:
        return math.nan
    parts = s.strip().split(":")
    try:
        nums = list(map(int, parts))
    except Exception:
        return math.nan
    if len(nums) == 2:  # MM:SS
        mm, ss = nums
        return mm*60 + ss
    if len(nums) == 3:  # HH:MM:SS
        hh, mm, ss = nums
        return hh*3600 + mm*60 + ss
    return math.nan

def normalize_duration_seconds(row) -> float:
    v = row.get("speaking_duration_raw")
    if isinstance(v, (int, float)):  # numeric seconds
        return float(v)
    if isinstance(v, str):
        v = v.strip()
        if re.fullmatch(r"\d+(\.\d+)?", v):
            return float(v)
        sec = _parse_hhmm_or_mmss_to_seconds(v)
        if sec == sec:
            return sec
    # fallback from start/end
    start = row.get("start_time"); end = row.get("end_time")
    if isinstance(start, str) and isinstance(end, str) and ":" in start and ":" in end:
        try:
            fmt = "%H:%M:%S" if len(start.split(":")) == 3 else "%H:%M"
            t0 = datetime.strptime(start, fmt)
            t1 = datetime.strptime(end, fmt)
            delta = (t1 - t0).total_seconds()
            if delta >= 0:
                return float(delta)
        except Exception:
            pass
    return math.nan

df["duration_sec"] = df.apply(normalize_duration_seconds, axis=1)

In [5]:
MIN_DURATION_SEC = 15
EXCLUDE_NONE = True

# Keep only rows with at least one code
df = df[df["codes"].apply(lambda lst: isinstance(lst, list) and len(lst) > 0)].copy()

# Drop "None" code entirely
if EXCLUDE_NONE:
    df["codes"] = df["codes"].apply(lambda lst: [c for c in lst if c != "None"])
    df = df[df["codes"].apply(lambda lst: len(lst) > 0)].copy()

# Keep only duration >= 15s
df = df[df["duration_sec"].fillna(0) >= MIN_DURATION_SEC].copy()

print("After filters — rows:", len(df))
from collections import Counter
avail_post = Counter(c for lst in df["codes"] for c in lst)
print("Available per code (post-filters):", dict(sorted(avail_post.items())))

After filters — rows: 10455
Available per code (post-filters): {'Coordination and Decision Practices': 1402, 'Evaluation Practices': 2119, 'Idea Management': 4662, 'Information Seeking': 2600, 'Integration Practices': 817, 'Knowledge Sharing': 7425, 'Participation Dynamics': 765, 'Relational Climate': 1749}


In [6]:
import random
from collections import Counter

def balanced_sample_strict(df, samples_per_code: int, seed: int):
    rng = random.Random(seed)
    work = df.copy()
    work["codes"] = work["codes"].apply(lambda x: x if isinstance(x, list) else [])
    all_codes = sorted({c for lst in work["codes"] for c in lst})
    if not all_codes:
        return work.sample(n=min(samples_per_code, len(work)), random_state=seed)

    codes_by_row = {i: set(lst) for i, lst in work["codes"].items()}
    idxs_by_code = {c: work.index[work["codes"].apply(lambda lst: c in lst)].tolist()
                    for c in all_codes}
    for c in all_codes:
        rng.shuffle(idxs_by_code[c])

    need = {c: samples_per_code for c in all_codes}
    selected, pool = set(), set().union(*idxs_by_code.values())

    def row_score(i):
        # +2 if helps underrepresented codes, -1 if adds to satisfied ones
        return sum(2 if need.get(c, 0) > 0 else -1 for c in codes_by_row[i])

    while any(n > 0 for n in need.values()) and pool:
        for c in sorted(need, key=lambda k: need[k], reverse=True):
            if need[c] <= 0:
                continue
            candidates = [i for i in idxs_by_code[c] if i in pool and i not in selected]
            if not candidates:
                continue
            candidates.sort(key=lambda i: (row_score(i), -len(codes_by_row[i])), reverse=True)
            pick = candidates[0]
            selected.add(pick); pool.remove(pick)
            for cc in codes_by_row[pick]:
                if need.get(cc, 0) > 0:
                    need[cc] -= 1

        # stop if remaining rows can’t help any needed code
        if not any(i in pool and any(need.get(cc, 0) > 0 for cc in codes_by_row[i]) for i in pool):
            break

    sampled = work.loc[sorted(selected)].copy().reset_index(drop=True)
    per_code = Counter(c for lst in sampled["codes"] for c in lst)
    print("Target per code:", samples_per_code)
    print("Final per-code counts:", dict(sorted(per_code.items())))
    unmet = {c: n for c, n in need.items() if n > 0}
    if unmet:
        print("Could not reach target for some codes (too rare):", unmet)
    return sampled

In [7]:
PCT_OF_DATA = 0.10
num_rows_target = max(1, int(round(PCT_OF_DATA * len(df))))
avg_labels = df["codes"].apply(len).mean() if len(df) else 1.0
num_codes = len({c for lst in df["codes"] for c in lst})

# total desired appearances ≈ rows_target * avg_labels
total_code_appearances_target = int(round(num_rows_target * avg_labels))
per_code_target_raw = max(1, total_code_appearances_target // max(1, num_codes))

available_by_code = Counter(c for lst in df["codes"] for c in lst)
SAMPLES_PER_CODE = max(1, min(per_code_target_raw, min(available_by_code.values())))

print(f"Rows target (~10%): {num_rows_target}")
print(f"Avg labels/row: {avg_labels:.2f}")
print(f"Per-code target (capped): {SAMPLES_PER_CODE}")

sampled = balanced_sample_strict(df, SAMPLES_PER_CODE, RANDOM_SEED)

Rows target (~10%): 1046
Avg labels/row: 2.06
Per-code target (capped): 269
Target per code: 269
Final per-code counts: {'Coordination and Decision Practices': 271, 'Evaluation Practices': 269, 'Idea Management': 290, 'Information Seeking': 271, 'Integration Practices': 269, 'Knowledge Sharing': 306, 'Participation Dynamics': 269, 'Relational Climate': 269}


In [8]:
from collections import Counter
per_code = Counter(c for lst in sampled["codes"] for c in lst)
print("Per-code counts in FINAL SAMPLE:", dict(sorted(per_code.items())))
print("Rows in FINAL SAMPLE:", len(sampled))

assert len(sampled) <= len(df)  # simple guardrail
sampled.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)

Per-code counts in FINAL SAMPLE: {'Coordination and Decision Practices': 271, 'Evaluation Practices': 269, 'Idea Management': 290, 'Information Seeking': 271, 'Integration Practices': 269, 'Knowledge Sharing': 306, 'Participation Dynamics': 269, 'Relational Climate': 269}
Rows in FINAL SAMPLE: 776
Wrote: /Users/maxchalekson/Desktop/gemini_data_analysis/sampling/sample_balanced.csv


In [12]:
print("Min duration in sample:", sampled["duration_sec"].min())
print("Max duration in sample:", sampled["duration_sec"].max())
sampled.sort_values("duration_sec").head(10)[["duration_sec", "transcript"]]

Min duration in sample: 15.0
Max duration in sample: 347.0


,duration_sec,transcript
522,15.0,I I I try to um yeah I I I try to put in appro...
675,15.0,But then um a pretty big uh discussion of of w...
683,15.0,And so that was perfect and I and and as you d...
335,15.0,"Yeah, I think that underscores the point the r..."
187,15.0,Essentially we talked about benefits of studyi...
398,15.0,"Oh, I've been talking away and I've been muted..."
656,15.0,"Well, that was that was a great discussion. Th..."
655,15.0,"Yeah, I think we're going to all um apparate s..."
197,15.0,Yeah I just want to add that I think it's grea...
179,15.0,"Great, thank you everyone for the introduction..."


In [13]:
num_short = (sampled["duration_sec"] < 15).sum()
print("Utterances in sample < 15s:", num_short)

Utterances in sample < 15s: 0


## quick check - verifying 10% amount sampled

In [9]:
# Size of dataset before sampling (after filters but before balancing)
N_filtered = len(df)

# Size of sampled set
N_sampled = len(sampled)

# % of dataset you actually sampled
pct = 100.0 * N_sampled / N_filtered if N_filtered else 0

print(f"Filtered dataset size (eligible utterances): {N_filtered}")
print(f"Sampled size: {N_sampled}")
print(f"That’s {pct:.1f}% of the eligible dataset.")

# Also show per-code counts
from collections import Counter
per_code = Counter(c for lst in sampled["codes"] for c in lst)
print("\nPer-code counts in sampled set:")
for k in sorted(per_code):
    print(f"{k}: {per_code[k]}")

Filtered dataset size (eligible utterances): 10455
Sampled size: 776
That’s 7.4% of the eligible dataset.

Per-code counts in sampled set:
Coordination and Decision Practices: 271
Evaluation Practices: 269
Idea Management: 290
Information Seeking: 271
Integration Practices: 269
Knowledge Sharing: 306
Participation Dynamics: 269
Relational Climate: 269


## ~20 per code

In [10]:
# === Alternative: fixed ~20 per code ===

SAMPLES_PER_CODE = 20  # Evey's ballpark target
print(f"\nBuilding fixed ~{SAMPLES_PER_CODE} per code sample...")

sampled_20 = balanced_sample_strict(df, SAMPLES_PER_CODE, RANDOM_SEED)

# Quick check
from collections import Counter
per_code_20 = Counter(c for lst in sampled_20["codes"] for c in lst)
print("Per-code counts in 20-sample:", dict(sorted(per_code_20.items())))
print("Rows in 20-sample:", len(sampled_20))

# Save separately so you keep both versions
OUT_CSV_20 = BASE_DIR.parent / "sampling" / "sample_balanced_20.csv"
sampled_20.to_csv(OUT_CSV_20, index=False)
print("Wrote:", OUT_CSV_20)


Building fixed ~20 per code sample...
Target per code: 20
Final per-code counts: {'Coordination and Decision Practices': 20, 'Evaluation Practices': 20, 'Idea Management': 20, 'Information Seeking': 20, 'Integration Practices': 20, 'Knowledge Sharing': 20, 'Participation Dynamics': 20, 'Relational Climate': 20}
Per-code counts in 20-sample: {'Coordination and Decision Practices': 20, 'Evaluation Practices': 20, 'Idea Management': 20, 'Information Seeking': 20, 'Integration Practices': 20, 'Knowledge Sharing': 20, 'Participation Dynamics': 20, 'Relational Climate': 20}
Rows in 20-sample: 54
Wrote: /Users/maxchalekson/Desktop/gemini_data_analysis/sampling/sample_balanced_20.csv
